# Optimizing Model Parameters

## 학습 내용
- Loss Function
- Optimizer
- Gradient Descent
- Training Loop
- optimizer.zero_grad()
- loss.backward()
- optimizer.step()
- Train / Validation / Test 과정

In [3]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

training_data = datasets.FashionMNIST(
    root="../data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

test_data = datasets.FashionMNIST(
    root="../data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

train_dataloader = DataLoader(training_data, batch_size=64)
test_dataloader = DataLoader(test_data, batch_size=64)

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28 * 28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )
    
    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork()


## Hyperparameters

- Epochs: 전체 데이터셋을 몇 번 반복 학습할지 결정하는 값
- Batch Size: 한 번의 학습에서 모델에 입력되는 데이터 샘플 수
- Learning Rate: Gradient Descent에서 매개변수(weight, bias)를 얼마나 크게 업데이트할지 결정하는 값

In [6]:
learning_rate = 1e-3
batch_size = 64
epochs = 5

## Optimization Loop

- Epoch는 Train Loop와 Validation/Test Loop로 구성된다.
- Train Loop는 모델을 학습하고 파라미터를 업데이트한다.
- Validation/Test Loop는 모델 성능을 평가한다.

## Loss Function

- Loss는 예측값과 실제값의 차이를 나타낸다.
- 학습의 목표는 Loss를 최소화하는 것이다.
- 회귀 문제는 주로 MSELoss를 사용한다.
- 분류 문제는 주로 CrossEntropyLoss를 사용한다.
- CrossEntropyLoss는 내부적으로 Softmax를 포함하므로 학습 시 logits을 그대로 입력한다.

In [ ]:
# 분류 문제를 위한 손실 함수 생성
# CrossEntropyLoss = LogSoftmax + NLLLoss
# 정답 클래스의 확률이 높을수록 Loss가 작아짐
loss_fn = nn.CrossEntropyLoss()

## Optimizer

- Optimizer는 loss를 줄이기 위해 모델의 parameter를 업데이트한다.
- SGD, Adam, RMSProp 등의 다양한 optimizer가 존재한다.
- SGD: 확률적 경사하강법 매 step마다 일정한 학습률로 업데이트
- Adam: 관성(momentum) + 파라미터별 학습률 자동 조정, SGD보다 안정적으로 수렴
- RMSProp: 파라미터별 학습률 자동 조정, 주로 RNN에서 사용

In [8]:
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

## Training Step

- optimizer.zero_grad(): 이전 gradient 초기화
- loss.backward(): gradient 계산
- optimizer.step(): parameter 업데이트

## 핵심 수식
- W = W - lr * dLoss/dW

## Full Implementation

- train_loop: 각 batch마다 forward → loss 계산 → backward → optimizer.step()을 수행하여 모델을 학습시킨다. (optimizer.zero_grad()로 gradient 초기화 포함)
- test_loop: torch.no_grad()를 사용해 gradient 계산을 끄고, 모델의 일반화 성능을 평가한다.

In [ ]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    # 모델을 train 모드로 설정
    model.train()

    # batch 단위로 학습
    for batch, (X, y) in enumerate(dataloader):
        pred = model(X)
        # loss 계산
        loss = loss_fn(pred, y)

        # Backpropagation (gradient 계산)
        loss.backward()
        # Parameter 업데이트
        optimizer.step()
        # Gradient 초기화 (다음 batch 위해 필수)
        optimizer.zero_grad()

        # 100 batch마다 로그 출력
        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f}    [{current:>5d}/{size:>5d}]")

def test_loop(dataloader, model, loss_fn):
    # 평가 모드 (Dropout, BatchNorm 비활성화)
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    # gradient 계산 비활성화 (메모리 절약 + 속도 향상)
    with torch.no_grad():
        for X, y in dataloader:
            # Forward pass
            pred = model(X)
            # Loss 누적
            test_loss += loss_fn(pred, y).item()
            # 정확도(Accuracy) 계산
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    # 평균 loss
    test_loss /= num_batches
    # 정확도
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [ ]:
loss_fn = nn.CrossEntropyLoss()

# 확률적 경사하강법(SGD) optimizer 생성
# model.parameters(): 학습할 weight, bias 전체
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

epochs = 10

for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    # 학습 단계 (weight 업데이트 발생)
    train_loop(train_dataloader, model, loss_fn, optimizer)
    
     # 평가 단계 (weight 업데이트 없음)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.304769    [   64/60000]
loss: 2.291167    [ 6464/60000]
loss: 2.270768    [12864/60000]
loss: 2.264065    [19264/60000]
loss: 2.260890    [25664/60000]
loss: 2.224829    [32064/60000]
loss: 2.233612    [38464/60000]
loss: 2.200441    [44864/60000]
loss: 2.192586    [51264/60000]
loss: 2.166167    [57664/60000]
Test Error: 
 Accuracy: 53.1%, Avg loss: 2.162732 

Epoch 2
-------------------------------
loss: 2.177308    [   64/60000]
loss: 2.163389    [ 6464/60000]
loss: 2.108692    [12864/60000]
loss: 2.119259    [19264/60000]
loss: 2.076513    [25664/60000]
loss: 2.017910    [32064/60000]
loss: 2.044714    [38464/60000]
loss: 1.968783    [44864/60000]
loss: 1.959420    [51264/60000]
loss: 1.893131    [57664/60000]
Test Error: 
 Accuracy: 57.8%, Avg loss: 1.891854 

Epoch 3
-------------------------------
loss: 1.933972    [   64/60000]
loss: 1.891703    [ 6464/60000]
loss: 1.778064    [12864/60000]
loss: 1.810646    [19264/60000]
loss: 1.

## 정리

- CrossEntropy는 분류 문제에서 사용하는 대표적인 손실 함수이다.
- loss_fn은 모델의 출력(logits)과 정답(label)을 비교하여 loss를 계산한다.
- optimizer는 loss를 줄이기 위해 model의 parameter(weight, bias)를 업데이트한다.
- SGD는 가장 기본적인 optimizer로 gradient를 이용해 weight를 갱신한다.

## 핵심 흐름

- forward: 모델 예측
- loss: 오차 계산
- backward: gradient 계산
- step: parameter 업데이트
- zero_grad: gradient 초기화

## 전체 학습 과정

- 데이터 → 모델 → loss → backward → optimizer → weight 업데이트 → 반복